<a href="https://colab.research.google.com/github/scelliot-cpu/Elliott_DSPN_26/blob/master/ExerciseSubmissions/15_power-analysis-via-simulations.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Exercise 15: Power analyses

This  assignment is designed to give you practice with Monte Carlo methods to conduct power analyses via simulation. You won't need to load in any data for this homework. We will, however, be using parts of the homework from last week.

---
## 1. Simulating data (1 point)


Pull your `simulate_data()` function from your last homework and add it below.

As a reminder, this function simulates the relationship between age, word reading experience, and reading comprehension skill.

`c` is reading comprehension, and `x` is word reading experience.

In [1]:
sample_size = 100 # How many children in data set?
age_lo = 80     # minimum age, in months
age_hi = 200    # maximum age, in months
beta_xa = 0.5   # amount by which experience changes for increase of one month in age
beta_x0 = -5    # amount of experience when age = 0 (not interpretable, since minimum age for this data is 80 months)
sd_x = 50       # standard dev of gaussian noise term, epsilon_x
beta_ca = 0.8   # amount that comprehension score improves for every increase of one unit in age
beta_cx = 3     # amount that comprehension score improves for every increase of one unit in reading experience
beta_c0 = 10    # comprehension score when reading experience is 0.
sd_c = 85      # standard dev of gaussian noise term, epsilon_c

simulate_data <- function(sample_size, age_lo, age_hi, beta_xa, beta_x0, sd_x, beta_ca, beta_cx, beta_c0, sd_c) {
      age <- runif(sample_size, min = age_lo, max = age_hi)
      epsilon_x <- rnorm(sample_size, mean = 0, sd = sd_x)
      x <- beta_xa * age + beta_x0 + epsilon_x
      epsilon_c <- rnorm(sample_size, mean = 0, sd = sd_c)
      c <- beta_ca * age + beta_cx * x + beta_c0 + epsilon_c
      dat <- data.frame(age = age, experience = x, comprehension = c)

      return(data.frame(age=age,x=x,c=c)) # it's actually bad form to have a variable named "c" in R, my bad...
}

dat <- simulate_data(sample_size, age_lo, age_hi, beta_xa, beta_x0, sd_x, beta_ca, beta_cx, beta_c0, sd_c)
head(dat)

,age,x,c
,<dbl>,<dbl>,<dbl>
1,129.75432,36.92832,149.3586
2,155.66738,195.26914,571.9026
3,160.92415,62.60738,370.9857
4,134.97851,125.45675,538.0207
5,82.67922,25.13791,220.2071
6,124.97027,49.17004,235.4996


---
## 2. `run_analysis()` function (2 points)

Last week, we looked at whether word reading experience(`x`) mediated the relation between `age` and reading comprehension (`c`).

Now we're going to use our `simulate_data()` function to conduct a power analysis. The goal is to determine how many participants we would need in order to detect both the mediated and the direct effects in this data.

*Note: We're going to pretend for the sake of simplicity that we don't have any control over the ages of the children we get (so ages are generated using `runif(sample_size, age_lo, age_hi)`, although of course this would be an unusual situation in reality.*

First, write a function, `run_analysis()`, that takes in simulated data, runs **your mediation from last week**, and returns a vector containing the ACME and ADE estimates and p-values (these are the `d0`, `d0.p`, `z0`, and `z0.p` features of the mediated model object, e.g., `fitMed$d0.p`). Print this function's output for the data we simulated previously.

In [8]:
library(mediation)
run_analysis <- function(dat) {
  fitM <- lm(x ~ age, data = dat)
  fitY <- lm(c ~ age + x, data = dat)
  fitMed <- mediate(fitM, fitY, treat = "age", mediator = "x")

  return(c(
  d0 = fitMed$d0,
  d0.p = fitMed$d0.p,
  z0 = fitMed$z0,
  z0.p = fitMed$z0.p))
}
run_analysis(dat)

d0      d0.p        z0      z0.p 
1.6172608 0.0000000 0.7829504 0.0000000

---
## 3. `repeat_analysis()` function (3 points)

Next fill in the function `repeat_analysis()` below so that it simulates and analyzes data `num_simulations` times. Store the outputs from each simulation in the `simouts` matrix. Calculate and return the coverage across all the simulations run for both ACME and ADE.

In [12]:
repeat_analysis <- function(num_simulations, alpha, sample_size, age_lo, age_hi,
        beta_xa, beta_x0, sd_x, beta_ca, beta_cx, beta_c0, sd_c) {
    # Initialize simouts matrix for storing each output from run_analysis()
    simouts <- matrix(rep(NA, num_simulations*4), nrow=num_simulations, ncol=4)

    # Start simulating
    for (i in 1:num_simulations) {
      dat <- simulate_data(
        sample_size, age_lo, age_hi, beta_xa, beta_x0, sd_x, beta_ca, beta_cx, beta_c0, sd_c
      )
      simouts[i,] <- run_analysis(dat)

    }

    # Calculate coverage for both ACME and ADE estimates using p-values in simouts
    ACME_cov = mean(simouts[,2] <= alpha)
    ADE_cov =  mean(simouts[,4] <= alpha)

    return(list(ACME_cov = ACME_cov, ADE_cov = ADE_cov))
}

Now run the `repeat_analysis()` function using the same parameter settings as above, for 10 simulations, with an alpha criterion of 0.01.

In [13]:
repeat_analysis(num_simulations = 10, alpha = 0.01, sample_size = sample_size, age_lo = age_lo, age_hi = age_hi, beta_xa = beta_xa, beta_x0 = beta_x0, sd_x = sd_x, beta_ca = beta_ca, beta_cx = beta_cx, beta_c0 = beta_c0, sd_c = sd_c)

$ACME_cov
[1] 0.7

$ADE_cov
[1] 0.6

---
## 4. Testing different sample sizes (2 points)

Finally, do the same thing (10 simulations, alpha criterion of 0.01) but for 5 different sample sizes: 50, 75, 100, 125, 150. You can do this using `map` (as in the tutorial), or a simple `for` loop, or by calculating each individually. Up to you! This should take around 3 minutes to run.

In [14]:
sample_sizes <- c(50, 75, 100, 125, 150)

results <- list()

for (n in sample_sizes) {
  results[[as.character(n)]] <- repeat_analysis(
    num_simulations = 10,
    alpha = 0.01,
    sample_size = n,
    age_lo = age_lo,
    age_hi = age_hi,
    beta_xa = beta_xa,
    beta_x0 = beta_x0,
    sd_x = sd_x,
    beta_ca = beta_ca,
    beta_cx = beta_cx,
    beta_c0 = beta_c0,
    sd_c = sd_c
  )
}


Print your results.

In [15]:
print(results)

$`50`
$`50`$ACME_cov
[1] 0.3

$`50`$ADE_cov
[1] 0.5


$`75`
$`75`$ACME_cov
[1] 0.7

$`75`$ADE_cov
[1] 0.5


$`100`
$`100`$ACME_cov
[1] 0.9

$`100`$ADE_cov
[1] 0.7


$`125`
$`125`$ACME_cov
[1] 0.9

$`125`$ADE_cov
[1] 0.6


$`150`
$`150`$ACME_cov
[1] 0.8

$`150`$ADE_cov
[1] 0.8




## 5. Reflection (2 pts)

If this were a real power analysis, we'd want to run more simulations per sample size (to get a more precise estimate of power) and we may also want to test out some other values of the parameters we used to simulate our data. However, what would you conclude just based on the results above?

> *Based on the results above, we can see that as sample size increased so did the values for the ACME, which is the mediated effect, and ADE, which is the direct effect. Additionally, the mediated effect is detected more consistently, so this means we might need larger sample sizes to detect smaller effects.*
>

Given how we generated the data, why was the direct effect harder to detect than the mediated effect?
> *Based on how we set up "comprehension ~ age + experience," the relationship between age and comprhension also goes through experience. Because experience is included in the model, that leaves only a small effects of age that isn't included in comprehension.*

**DUE:** 11:59pm EST, March 31, 2026

**IMPORTANT** Did you collaborate with anyone on this assignment? If so, list their names here.
> *Someone's Name*

**GenAI Utilization** Did you utilize any generative AI tools on this assignment? If so, please list the item and the paste respective prompt you used.

> ChatGPT: why is the function outputing TRUE or FALSE for ACME and ADE instead of a proportion? *pasted code from calculating coverage, it was a parentheses in the wrong spot
>